# Countdown + `dim_site` Crosswalk

Two Week-1 artifacts in one place:

- **A — Countdown / feasibility baseline.** Backlog vs. the 9/30 goal: required go-live rate, the
  target-date schedule (how much is already late or scheduled to miss), and — when an Airtable token
  is present — trailing actual velocity and the resulting gap. This is the scoreboard.
- **B — `dim_site` crosswalk.** The conformed site key: one row per site service, with the SFDC and
  Airtable identifiers lined up. The seed of the lexicon / SSOT spine.

Shared loaders and business rules live in `recon_lib.py`. The SFDC side runs from the CSV alone;
Airtable-dependent cells light up once `AIRTABLE_API_TOKEN` is in `.env`.

In [ ]:
import recon_lib as rl
import pandas as pd, datetime as dt
pd.set_option("display.max_columns", None, "display.width", 200)

SFDC_CSV = "data/sfdc_report_2026-06-22.csv"
TODAY = dt.date.today()

sfdc = rl.load_sfdc(SFDC_CSV)
at   = rl.load_airtable()                 # None if no token in .env
print(f"SFDC backlog rows: {len(sfdc)}   Airtable: {'loaded ('+str(len(at))+')' if at is not None else 'no token — AT cells will skip'}")

## A — Countdown / feasibility baseline

The backlog here is the focused, signed-not-live population (the SFDC report's contents). Required
rate = remaining ÷ weeks to 9/30.

In [ ]:
GOAL = rl.GOAL_DATE
weeks_left = max((GOAL - TODAY).days / 7, 0.1)

backlog = {
    "Site services": len(sfdc),
    "Locations":     sfdc["address"].map(rl.norm_text).replace("", pd.NA).nunique(),
    "Jobs":          sfdc["job_id_key"].nunique(),
    "Accounts":      sfdc["account"].replace("", pd.NA).nunique(),
    "Sqft":          int(sfdc["sqft_num"].sum()),
}
required_loc_wk = backlog["Locations"] / weeks_left
print(f"Backlog: {backlog}")
print(f"Weeks to {GOAL}: {weeks_left:.1f}")
print(f"REQUIRED RATE: {required_loc_wk:.0f} locations/week ({backlog['Site services']/weeks_left:.0f} site services/week)")

### Target-go-live schedule — what's already late or scheduled to miss

This is the "never surprised" cut: the plan vs. the goal date, *before* anyone touches velocity.

In [ ]:
tg = sfdc["target_golive_d"]
sched = {
    "Already past target (late)": int((tg.notna() & (tg < TODAY)).sum()),
    "Targeted on/before 9/30":    int((tg.notna() & (tg >= TODAY) & (tg <= GOAL)).sum()),
    "Targeted AFTER 9/30 (will miss)": int((tg.notna() & (tg > GOAL)).sum()),
    "No target date":             int(tg.isna().sum()),
}
sched_df = pd.DataFrame.from_dict(sched, orient="index", columns=["site_services"])
sched_df["pct"] = (sched_df["site_services"] / len(sfdc) * 100).round(1)
at_risk = sched["Already past target (late)"] + sched["Targeted AFTER 9/30 (will miss)"]
print(f"AT-RISK (already late OR scheduled past 9/30): {at_risk} of {len(sfdc)} "
      f"({at_risk/len(sfdc)*100:.0f}%)")
sched_df

### Trailing actual velocity + the gap *(requires Airtable token)*

Counts real go-lives per ISO week from `Actual Go-Live Date` on the live jobs, takes the trailing
rate, and compares it to the required rate. The gap is the headline number for Anil.

In [ ]:
if at is None:
    print("No Airtable token — set AIRTABLE_API_TOKEN in .env to compute trailing velocity.")
else:
    live = at[at["actual_golive_d"].notna()].copy()
    live["wk"] = live["actual_golive_d"].map(lambda d: d.isocalendar()[:2])
    cutoff = TODAY - dt.timedelta(weeks=8)
    recent = live[live["actual_golive_d"] >= cutoff]
    by_wk = recent.groupby("wk").size().sort_index()
    trailing_rate = len(recent) / 8
    print("Go-lives per week (last 8 wks):")
    print(by_wk.to_string())
    print(f"\nTrailing actual rate : {trailing_rate:.0f} jobs/week")
    print(f"Required rate        : {required_loc_wk:.0f} locations/week")
    gap = required_loc_wk - trailing_rate
    verdict = "ON TRACK" if gap <= 0 else f"SHORT by {gap:.0f}/week -> ~{gap*weeks_left:.0f} locations at risk"
    print(f"GAP                  : {verdict}")

In [ ]:
# Persist the countdown snapshot (trend line for the scoreboard)
import os
os.makedirs("runs", exist_ok=True)
snap = {"date": TODAY.isoformat(), "weeks_left": round(weeks_left, 1),
        "required_loc_per_wk": round(required_loc_wk, 1), **backlog, **sched}
path = "runs/countdown_history.csv"
pd.DataFrame([snap]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
print("Appended countdown snapshot ->", path)
pd.DataFrame([snap]).T

## B — `dim_site` crosswalk (the conformed site key)

One row per SFDC site service, with the Airtable record lined up via the job-id join. This is the
shared key (`dim_site`) other models reconcile against — published as `runs/dim_site.csv`.

In [ ]:
cross = sfdc[["site_service_id", "site_service", "job_id_key", "job_name",
              "account", "address", "owner", "sqft_num", "job_status",
              "job_type", "target_golive"]].copy()
cross = cross.rename(columns={"job_id_key": "sfdc_18char_job_id", "sqft_num": "sqft"})

if at is not None:
    at_keyed = (at.dropna(subset=["job_id_key"])
                  .drop_duplicates("job_id_key")
                  .set_index("job_id_key"))
    cross["at_record_id"]    = cross["sfdc_18char_job_id"].map(at_keyed["at_record_id"])
    cross["at_account"]      = cross["sfdc_18char_job_id"].map(at_keyed["account"])
    cross["at_job_status"]   = cross["sfdc_18char_job_id"].map(at_keyed["job_status"])
    matched = cross["at_record_id"].notna().sum()
    print(f"Crosswalk: {len(cross)} SFDC sites | {matched} matched to Airtable "
          f"({matched/len(cross)*100:.0f}%) | {len(cross)-matched} SFDC-only")
else:
    cross["at_record_id"] = pd.NA
    print(f"Crosswalk: {len(cross)} SFDC sites (Airtable columns blank — no token)")

cross.to_csv("runs/dim_site.csv", index=False)
print("Wrote runs/dim_site.csv")
cross.head(10)